# Depdencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import Subset

import numpy as np
import matplotlib.pyplot as plt
# import joblib

import os
# import zipfile
from pathlib import Path

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

# DuoCL

In [ ]:
class duoCL(nn.Module):
    def __init__(self):
        super(duoCL, self).__init__()
        
        # === 1st Conv path with kernel = 3 ===
        self.conv_13 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1)
        self.relu_11 = nn.ReLU()
        self.pool_11 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_23 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1)
        self.relu_12 = nn.ReLU()
        self.pool_12 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_33 = nn.Conv1d(in_channels=32, out_channels=128, kernel_size=3, stride=1)
        self.relu_13 = nn.ReLU()
        self.pool_13 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_43 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, stride=1)
        self.relu_14 = nn.ReLU()
        self.pool_14 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_53 = nn.Conv1d(in_channels=256, out_channels=512, kernel_size=3, stride=1)
        self.dropout_1 = nn.Dropout(0.5)
        
        self.flatten_3 = nn.Flatten()
        
        self.fc_1 = nn.Linear(in_features=14336,out_features=1024)
        self.fc_2 = nn.Linear(in_features=1024,out_features=512)
        
        # === 2nd Conv path with kernel = 7 ===
        self.conv_17 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=7, stride=1)
        self.relu_21 = nn.ReLU()
        self.pool_21 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_27 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, stride=1)
        self.relu_22 = nn.ReLU()
        self.pool_22 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_37 = nn.Conv1d(in_channels=32, out_channels=128, kernel_size=7, stride=1)
        self.relu_23 = nn.ReLU()
        self.pool_23 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_47 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=7, stride=1)
        self.relu_24 = nn.ReLU()
        self.pool_24 = nn.AvgPool1d(kernel_size=2, stride=2)
        
        self.conv_57 = nn.Conv1d(in_channels=256, out_channels=512, kernel_size=7, stride=1)
        self.dropout_2 = nn.Dropout(0.5)
        
        self.flatten_4 = nn.Flatten()

        self.fc_3 = nn.Linear(in_features=10240,out_features=1024)
        self.fc_4 = nn.Linear(in_features=1024,out_features=512)
        
        # === Separate LSTMs ===
        self.lstm_1 = nn.LSTM(input_size=512, hidden_size=512, num_layers=3, batch_first=False)
        self.lstm_2 = nn.LSTM(input_size=512, hidden_size=512, num_layers=3, batch_first=False)
        
        # FC Layer after LSTM
        self.fc_5 = nn.Linear(in_features=1024, out_features=512)
        
    def forward(self, x):
        # --- Path 1 (kernel=3) ---
        x1 = self.pool_11(self.relu_11(self.conv_13(x)))
        x1 = self.pool_12(self.relu_12(self.conv_23(x1)))
        x1 = self.pool_13(self.relu_13(self.conv_33(x1)))
        x1 = self.pool_14(self.relu_14(self.conv_43(x1)))
        x1 = self.dropout_1(self.conv_53(x1))   # Shape: (B, 512, L)
        
        # Prepare for LSTM: transpose to (B, L1, 512)
        x1 = x1.permute(0, 2, 1)  # (B, L1, C)
        out1, _ = self.lstm_1(x1)  # (B, L1, 512)
        out1 = out1[:, -1, :]      # Take last time step (B, 512)

        # --- Path 2 (kernel=7) ---
        x2 = self.pool_21(self.relu_21(self.conv_17(x)))
        x2 = self.pool_22(self.relu_22(self.conv_27(x2)))
        x2 = self.pool_23(self.relu_23(self.conv_37(x2)))
        x2 = self.pool_24(self.relu_24(self.conv_47(x2)))
        x2 = self.dropout_2(self.conv_57(x2))   # Shape: (B, 512, L2)
        
        # Prepare for LSTM: transpose to (B, L2, 512)
        x2 = x2.permute(0, 2, 1)  # (B, L2, C)
        out2, _ = self.lstm_2(x2)  # (B, L2, 512)
        out2 = out2[:, -1, :]      # Take last time step (B, 512)

        # Concatenate outputs from both LSTMs
        out = torch.cat([out1, out2], dim=1)  # (B, 1024)

        # Final FC layer
        out = self.fc_5(out)  # (B, 512)
        out = out.unsqueeze(1) # (B, 1, 512)

        return out

# Dataset & Dataloader

In [ ]:
class EEG_Dataset(Dataset):
    """
    A PyTorch Dataset for loading raw and clean EEG epoch pairs.
    It expects the following structure:
    - data/
        - raw/
            - raw_training_epochs/
                - subject1/
                    - c3_epoch0_raw.pt
        - clean/
            - clean_training_epochs/
                - subject1/
                    - c3_epoch0_clean.pt
    """
    def __init__(self, data_dir, split):
        """
        Args:
            data_dir (str): The path to the root 'data' directory.
            split (str): The dataset split (e.g., 'training_epochs', 'validation_epochs', or 'test_epochs').
        """
        # Construct the paths to the raw and clean data folders for the specified split
        self.raw_dir = Path(data_dir) / 'raw' / f'raw_{split}'
        self.clean_dir = Path(data_dir) / 'clean' / f'clean_{split}'

        if not self.raw_dir.is_dir() or not self.clean_dir.is_dir():
            raise FileNotFoundError(f"One of the specified directories does not exist: {self.raw_dir} or {self.clean_dir}")

        self.file_pairs = []

        # Traverse the directory to find all raw files and their corresponding clean files
        for subject_dir in self.raw_dir.iterdir():
            if subject_dir.is_dir():
                for raw_file_path in subject_dir.glob('*.pt'):
                    # The clean file path is found by replacing the directory and file suffix
                    relative_path = raw_file_path.relative_to(self.raw_dir)
                    clean_file_path = self.clean_dir / relative_path.with_name(
                        raw_file_path.stem.replace('_raw', '_clean') + '.pt'
                    )

                    if clean_file_path.is_file():
                        self.file_pairs.append((raw_file_path, clean_file_path))
                    # else:
                    #     print(f"Warning: Corresponding clean file not found for {raw_file_path}")

    def __len__(self):
        """Returns the total number of data samples."""
        return len(self.file_pairs)

    def __getitem__(self, idx):
      """Loads and returns a raw and clean pair, normalized to [-1, 1]."""
      raw_path, clean_path = self.file_pairs[idx]

      # Load the tensors from their file paths
      raw_tensor = torch.load(raw_path)
      clean_tensor = torch.load(clean_path)

      # Add channel dim: [512] -> [1, 512]
      raw_tensor = raw_tensor.unsqueeze(0)
      clean_tensor = clean_tensor.unsqueeze(0)

      # Standardize each sample to [-1, 1]
      def normalize(tensor):
          min_val = tensor.min()
          max_val = tensor.max()
          if max_val > min_val:  # Avoid division by zero
              tensor = 2 * (tensor - min_val) / (max_val - min_val) - 1
          else:
              tensor = torch.zeros_like(tensor)
          return tensor

      raw_tensor = normalize(raw_tensor)
      clean_tensor = normalize(clean_tensor)

      return raw_tensor, clean_tensor



# Dataset & Loaders

# ===== Training =====
train_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme", 
    "training_epochs"
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


# ===== Valid =====
valid_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "validation_epochs"
)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=False)


# ===== Test =====
test_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "testing_epochs"
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Training

In [ ]:
# ===== Model, Loss, Optimizer =====
model = duoCL()
model = model.to(device)

criterion = nn.MSELoss()
learning_rate = 1e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

num_epochs = 200
train_loss_history = []
val_loss_history = []


# ===== Early Stopping Parameters =====
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None
patience = 20


# ===== Paths for saving =====
checkpoint_dir = '/kaggle/working/models'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'duocl_checkpoint.pth')

# ===== Resume if model checkpoint exists =====
start_epoch = 0
train_loss_history = []
val_loss_history = []

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_loss_history = checkpoint['train_loss_history']
    val_loss_history = checkpoint['val_loss_history']
    print(f"Resumed from epoch {start_epoch}")


# ===== Training =====
for epoch in range(start_epoch, num_epochs):
    model.train()
    train_running_loss = 0.0

    for batch_idx, (raw, clean) in enumerate(train_loader):
        raw = raw.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        outputs = model(raw)
        loss = criterion(outputs, clean)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()

    train_avg_loss = train_running_loss / len(train_loader)
    train_loss_history.append(train_avg_loss)

    # ====== Validation ======
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for val_raw, val_clean in valid_loader:
            val_raw = val_raw.to(device)
            val_clean = val_clean.to(device)

            val_outputs = model(val_raw)
            val_loss = criterion(val_outputs, val_clean)
            val_running_loss += val_loss.item()

    val_avg_loss = val_running_loss / len(valid_loader)
    val_loss_history.append(val_avg_loss)

    # ===== Early Stopping Check =====
    if val_avg_loss < best_val_loss:
        best_val_loss = val_avg_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_avg_loss:.6f} | Val Loss: {val_avg_loss:.6f} | No Improve: {epochs_no_improve}")

    if epochs_no_improve >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
        
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
            'hyperparameters': {
                'learning_rate': learning_rate,
                'num_epochs': num_epochs
            }
        }, checkpoint_path)
        print(f"Checkpoint saved to: {checkpoint_path}")



# ===== Saving model =====
model_dir = '/kaggle/working/models'
os.makedirs(model_dir, exist_ok=True)
model_cpu = model.to('cpu') # Switching back to cpu

# Save the model state dictionary
model_path = os.path.join(model_dir, 'duocl.pth')

# Save training history and hyperparameters
torch.save({
    'train_loss_history': train_loss_history,
    'val_loss_history': val_loss_history,
    'hyperparameters': {
        'learning_rate': learning_rate,
        'num_epochs': num_epochs
    }
}, model_path)
print(f"Training history saved to: {model_path}")

# Metrics

In [1]:
def compute_snr(raw, clean):
    noise = raw - clean
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    snr = 10 * np.log10(signal_power / noise_power)

    return snr

def compute_rrmse_time(raw, clean):
    numerator = np.sqrt(np.mean((raw - clean) ** 2))
    denominator = np.sqrt(np.mean(clean ** 2))
    return numerator / denominator

def compute_rrmse_freq(raw, clean):
    raw_fft = np.abs(np.fft.rfft(raw))
    clean_fft = np.abs(np.fft.rfft(clean))
    numerator = np.sqrt(np.mean((raw_fft - clean_fft) ** 2))
    denominator = np.sqrt(np.mean(clean_fft ** 2))
    return numerator / denominator

def compute_average_cc(raw, clean):
    correlations = []
    for i in range(raw.shape[0]):
        r = np.corrcoef(raw[i], clean[i])[0, 1]
        correlations.append(r)
    return np.mean(correlations)


def evaluate_model_metrics(model, dataloader):
    model.eval()
    snr_list = []
    rrmse_time_list = []
    rrmse_freq_list = []
    cc_list = []

    # Testing
    with torch.no_grad():
        for raw, clean in dataloader:
            # Run model forward
            model_output_batch = model(raw)

            # Move tensors to CPU and convert to numpy
            raw_np = raw.cpu().numpy()
            clean_np = clean.cpu().numpy()
            output_np = model_output_batch.cpu().numpy()

            # Compute metrics per sample in batch
            for i in range(raw_np.shape[0]):
                # raw_sample = raw_np[i]
                clean_sample = clean_np[i]
                model_output_sample = output_np[i]

                # Use model output and clean target (or raw if comparing raw to clean)
                snr_val = compute_snr(model_output_sample, clean_sample)
                rrmse_time_val = compute_rrmse_time(model_output_sample, clean_sample)
                rrmse_freq_val = compute_rrmse_freq(model_output_sample, clean_sample)
                cc_val = compute_average_cc(model_output_sample, clean_sample)

                snr_list.append(snr_val)
                rrmse_time_list.append(rrmse_time_val)
                rrmse_freq_list.append(rrmse_freq_val)
                cc_list.append(cc_val)

    # Aggregate metric results across entire dataset
    metrics = {
        "SNR": np.mean(snr_list),
        "RRMSE_time": np.mean(rrmse_time_list),
        "RRMSE_freq": np.mean(rrmse_freq_list),
        "Average_CC": np.mean(cc_list)
    }
    return metrics

# Testing

In [ ]:
# Printing metrics
metrics = evaluate_model_metrics(model, test_loader)

print("Test metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
    
with open('/kaggle/working/metrics_output.txt', 'w') as file:
    for metric, value in metrics.items():
        file.write(f"{metrics}: {value}")